# Phase 7: PatchCore Unsupervised Anomaly Detection
### Towards Total Recall in Industrial Quality Inspection (Roth et al., CVPR 2022)

**Core Objectives:**
1. Train an unsupervised one-class anomaly detection model on nominal (normal-only) industrial images.
2. Extract mid-level feature hierarchy representations (Layers 2 & 3) from frozen deep backbones (`wide_resnet50_2` / `resnet18`).
3. Apply **Greedy k-Center Coreset Subsampling** to compress memory bank prototypes by 90% while guaranteeing minimax boundary coverage.
4. Evaluate strictly on the official **MVTec AD Bottle** hold-out benchmark: Image AUROC, Pixel AUROC, and Per-Region Overlap (AUPRO).
5. Run comprehensive industrial ablations: Backbone, Coreset Ratio, Gaussian Smoothing, and Scoring Functions.
6. Upload validated checkpoints and telemetry to Hugging Face Hub.

## 1. Environment & Hardware Verification (Tesla T4 GPU)

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Dependencies & Repository Setup

In [ ]:
# Install scientific and computer vision packages
!pip install -q torchvision scikit-learn scipy opencv-python matplotlib huggingface_hub

## 3. Hugging Face Hub Authentication

In [ ]:
from google.colab import userdata
from huggingface_hub import login, HfApi

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=True)
    api = HfApi()
    print(f"Logged into Hugging Face Hub as: {api.whoami()['name']}")
except Exception as e:
    print(f"Note on Hugging Face authentication: {e}")

## 4. Run Baseline PatchCore Benchmark

In [ ]:
!python training/colab/anomaly/07_train_patchcore.py

## 5. Run Comprehensive Industrial Ablation Study
Ablations include:
- **Backbone:** ResNet-18 vs Wide-ResNet-50-2
- **Coreset Subsampling:** 5% vs 10% vs 20%
- **Spatial Filter:** Raw Bilinear (sigma=0.0) vs Gaussian-Smoothed (sigma=4.0)
- **Scoring Function:** Maximum Patch Distance vs Top-5 Mean Patch Distance

In [ ]:
!python src/training/run_patchcore_ablations.py --device cuda

## 6. Qualitative Visual Inspection & Ablation Charts

In [ ]:
from IPython.display import Image, display, Markdown

print("=== Inspection Heatmaps on MVTec Bottle ===")
display(Image("reports/figures/patchcore_mvtec_bottle_evaluation.png"))

print("=== Ablation Study Comparison Charts ===")
display(Image("reports/figures/patchcore_ablation_comparison.png"))

# Display markdown table summary
try:
    with open("reports/patchcore_ablation_summary.md", "r") as f:
        display(Markdown(f.read()))
except Exception as e:
    pass